<a href="https://colab.research.google.com/github/ZEELJARIWALA/AI-SOLAR/blob/main/medigpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tool

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.5 MB/s eta 0:00:00
  Created wheel for tool: filename=tool-0.8.0-py3-none-any.whl size=70551 sha256=88071f51ff1087c2aa5e023d8d47ea67c0440261e81181ffda5aba95f78b5f6c
  Stored in directory: /root/.cache/pip/wheels/a9/42/ff/06f0fd152b1b33ff7ec1d1ede211fd342e2d4edada6df3b798
Successfully built tool


In [3]:
!pip uninstall -y fitz # Uninstall the incorrect 'fitz' package
!pip install pymupdf  # Install PyMuPDF, which can be imported as 'fitz'
import fitz  # PyMuPDF

pdf_path = "/content/dataset_medi.pdf"

doc = fitz.open(pdf_path)


pages_text = []

for page in doc:
    text = page.get_text("text")
    pages_text.append(text)

full_text = "\n".join(pages_text)

# save raw
with open("charaka_raw.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

print("✅ Extraction done")
print("Total characters:", len(full_text))

✅ Extraction done
Total characters: 1115560


In [4]:
import re

with open("charaka_raw.txt", "r", encoding="utf-8") as f:
    text = f.read()

# remove page numbers as standalone lines (more robust)
# Matches lines that contain only digits, possibly surrounded by whitespace.
# (?m) enables multiline mode, so ^ and $ match start/end of each line.
text = re.sub(r'(?m)^\s*\d+\s*$', '', text)

# fix hyphen broken words (word- \n becomes word)
text = text.replace("-\n", "")
text = text.replace("-\r\n", "") # Handle Windows newlines if present

# remove multiple spaces and tabs
text = re.sub(r'[ \t]+', ' ', text)

# normalize newlines to at most two consecutive newlines
# This is applied again to ensure consistency after page number removal
text = re.sub(r'\n{3,}', '\n\n', text)

# remove weird unicode artifacts (optional but good)
text = text.replace('\xa0', ' ')

# strip leading/trailing whitespace from the entire text
text = text.strip()

with open("charaka_clean.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Cleaning done")

✅ Cleaning done


# Task
Download the missing NLTK resource `punkt_tab`, then re-run the semantic chunking code. After successfully chunking the document into `charaka_chunks.json`, install a Python library for generating embeddings, create numerical embeddings for each text chunk, and save these embeddings for future use.

## Download missing NLTK resource

### Subtask:
Add a line to download the `punkt_tab` resource using `nltk.download('punkt_tab')` to resolve the `LookupError`.


**Reasoning**:
The previous code block failed because the `punkt_tab` NLTK resource was not downloaded. I need to add `nltk.download('punkt_tab')` to ensure all necessary resources for `sent_tokenize` are available, and then re-run the chunking process.



In [6]:
import nltk
import re

nltk.download('punkt')
nltk.download('punkt_tab') # Download the missing resource

from nltk.tokenize import sent_tokenize

# load clean text
with open("charaka_clean.txt", "r", encoding="utf-8") as f:
    text = f.read()

# split into paragraphs first
paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

chunks = []
current_chunk = ""
max_words = 450  # sweet spot for 3B model

for para in paragraphs:
    words_in_para = len(para.split())
    words_in_chunk = len(current_chunk.split())

    # if paragraph fits, add directly
    if words_in_chunk + words_in_para <= max_words:
        current_chunk += " " + para

    else:
        # save previous chunk
        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        # if paragraph itself too big → sentence split
        if words_in_para > max_words:
            sentences = sent_tokenize(para)
            temp = ""

            for sent in sentences:
                if len((temp + sent).split()) <= max_words:
                    temp += " " + sent
                else:
                    chunks.append(temp.strip())
                    temp = sent

            current_chunk = temp

        else:
            current_chunk = para

# last chunk
if current_chunk.strip():
    chunks.append(current_chunk.strip())

print("✅ Total semantic chunks:", len(chunks))

# save chunks
import json

with open("charaka_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ Total semantic chunks: 484


In [7]:
lengths = [len(c.split()) for c in chunks]

print("Min words:", min(lengths))
print("Max words:", max(lengths))
print("Avg words:", sum(lengths)/len(lengths))


Min words: 16
Max words: 451
Avg words: 334.08677685950414


In [8]:
!pip install chromadb sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    

In [9]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

# 1️⃣ Load your chunks
with open("charaka_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Total chunks loaded:", len(chunks))

# 2️⃣ Load embedding model (this converts text → vectors)
embed_model = SentenceTransformer("BAAI/bge-small-en")

print(" Embedding model loaded")

# 3️⃣ Create vector database (Chroma)
client = chromadb.Client()
collection = client.create_collection(name="charaka_knowledge")

print(" Vector DB created")

# 4️⃣ Convert and store each chunk
for i, chunk in enumerate(chunks):
    embedding = embed_model.encode(chunk).tolist()

    collection.add(
        ids=[str(i)],
        embeddings=[embedding],
        documents=[chunk]
    )

    if i % 50 == 0:
        print(f"Stored {i} chunks...")

print("All chunks stored successfully!")


Total chunks loaded: 484


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Embedding model loaded
 Vector DB created
Stored 0 chunks...
Stored 50 chunks...
Stored 100 chunks...
Stored 150 chunks...
Stored 200 chunks...
Stored 250 chunks...
Stored 300 chunks...
Stored 350 chunks...
Stored 400 chunks...
Stored 450 chunks...
All chunks stored successfully!


In [10]:
print(collection.count())


484


In [11]:
def retrieve_context(query, k=3):
    # convert question → vector
    query_embedding = embed_model.encode(query).tolist()

    # search similar chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


In [12]:
question = "who is a author od charaksanhita?"

contexts = retrieve_context(question, k=5)

print("🔍 Top retrieved chunks:\n")

for i, ctx in enumerate(contexts):
    print(f"\n--- Chunk {i+1} ---\n")
    print(ctx[:500])


🔍 Top retrieved chunks:


--- Chunk 1 ---

The Treatise: 
 
 
 
 
 
Qualities of the Treatise: 
 
 
Various treaties in medicine are found in society. From amongst them, 
one should select that which is great, used by eminent and wise men, full of ideas, 
respected by authorities, intelligible and beneficial to all the 3 types of disciples (dull, 
mediocre and intelligent), free from the defect of repetition, coming down from the sages, 
with well-composed introduction, discussion and conclusion, having firm base, free from 
week and dif

--- Chunk 2 ---

Si12#52-54 
 
 
Thus ends ...the treatise composed by Agnivesa, redacted by Caraka and reconstructed by Drdhabala as it was not available. 
 
 
Si12#54 
 
 
[The above couple pages is information... About the Charaka Samhita: (from Si12#3454)] 
 
The end of the book (I/II): 
 
 
 
 
Thus the discourse of sage Atreya contained in 120 chapters has been delivered by wise 
Agnivesa for the well-being of all the people. 
 
One, by studying

In [13]:
for i, ctx in enumerate(contexts):
    print(f"\n===== RANK {i+1} =====")
    print("Word length:", len(ctx.split()))
    print(ctx[:400])  # show first 400 chars



===== RANK 1 =====
Word length: 393
The Treatise: 
 
 
 
 
 
Qualities of the Treatise: 
 
 
Various treaties in medicine are found in society. From amongst them, 
one should select that which is great, used by eminent and wise men, full of ideas, 
respected by authorities, intelligible and beneficial to all the 3 types of disciples (dull, 
mediocre and intelligent), free from the defect of repetition, coming down from the sages, 
w

===== RANK 2 =====
Word length: 344
Si12#52-54 
 
 
Thus ends ...the treatise composed by Agnivesa, redacted by Caraka and reconstructed by Drdhabala as it was not available. 
 
 
Si12#54 
 
 
[The above couple pages is information... About the Charaka Samhita: (from Si12#3454)] 
 
The end of the book (I/II): 
 
 
 
 
Thus the discourse of sage Atreya contained in 120 chapters has been delivered by wise 
Agnivesa for the well-being 

===== RANK 3 =====
Word length: 227
Charaka Samhita 
 
Handbook on Ayurveda 
 
Volume I 
 
 
 
 
Edited by Gabriel Van Loon

In [14]:
!pip install transformers accelerate sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 88.4 MB/s eta 0:00:00


In [16]:
!pip install -U transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import torch

model_name = "bharatgenai/AyurParam"

# Load config
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

# 🔥 FORCE SAFE VALUE
config.rope_scaling = None

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("✅ Model loaded safely!")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Model loaded safely!


In [18]:
def format_prompt(question, context=""):
    prompt = f"""<user>
Context:
{context}

Question:
{question}
</user>
<assistant>
"""
    return prompt


In [21]:
user_input = "Who is the author of Charaka Samhita?"
prompt = f"<user> {user_input} <assistant>"


In [22]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,   # ⭐ IMPORTANT: no sampling first
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Task
Fix the `KeyError: 'type'` encountered when loading the `bharatgenai/AyurParam` model by modifying its configuration to handle `rope_scaling` appropriately, then successfully load the model and proceed with the Retrieval Augmented Generation (RAG) implementation.

## Fix KeyError: 'type' in Model Configuration

### Subtask:
Modify the model loading code to explicitly handle the `rope_scaling` configuration. Load the configuration first, and if `rope_scaling` is present but missing the 'type' key, set it to `None` to ensure the model's custom code defaults to no scaling, thus resolving the `KeyError`.


**Reasoning**:
To resolve the `KeyError: 'type'` when loading the model, I need to explicitly load and modify the model's configuration. I will import `AutoConfig`, load the configuration, check for the `rope_scaling` attribute, and if it's a dictionary missing the 'type' key, set it to `None` before loading the model with the adjusted configuration.



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import torch

model_name = "bharatgenai/AyurParam"

# Load the configuration first
config = AutoConfig.from_pretrained(model_name)

# Check for rope_scaling and handle the missing 'type' key
if hasattr(config, 'rope_scaling') and isinstance(config.rope_scaling, dict) and 'type' not in config.rope_scaling:
    print("Warning: `rope_scaling` found without 'type' key. Setting `rope_scaling` to None.")
    config.rope_scaling = None

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config, # Pass the modified config
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True # Add this to avoid the prompt about custom code
)

print("Model loaded successfully!")

In [ ]:
def build_prompt(context_list, question):
    context_text = "\n\n".join(context_list)

    prompt = f"""
You are an expert Ayurvedic scholar.

Answer the question ONLY using the provided Charaka Samhita context.
If the answer is not present in the context, say:
"Not found in Charaka Samhita."

Context:
{context_text}

Question:
{question}

Answer:
"""
    return prompt


In [ ]:
prompt = "Who is the author of Charaka Samhita?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.3,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


# Task
# Plan:
* **Temporarily disable sampling**: Set `do_sample=False` in `model.generate`. This will force greedy decoding, bypassing the `torch.multinomial` step. If this resolves the error, it points to issues with the sampling parameters or the precision during sampling.
* **Add `pad_token_id` and `eos_token_id` explicitly**: Explicitly define `pad_token_id` and `eos_token_id` in the `model.generate` call to ensure consistent behavior, although the warning stated it was defaulted.
* **Adjust sampling parameters (if sampling is required)**: If `do_sample=True` is needed and the above changes don't fully resolve it, experiment with `temperature` (e.g., 0.7 to 1.0) and consider adding `top_p` (e.g., 0.9 or 0.95) or `top_k` to constrain the sampling space and potentially improve numerical stability with `float16`.
* **Consider higher precision (diagnostic)**: If numerical instability persists, as a diagnostic, consider reloading the model with `torch_dtype=torch.float32` to confirm if `float16` precision is the root cause. Be aware that `float32` requires more memory.

Fix the `RuntimeError: probability tensor contains either `inf`, `nan` or element < 0` during text generation by temporarily disabling sampling and explicitly setting `pad_token_id` and `eos_token_id` in the `model.generate` call.

## Temporarily disable sampling and explicitly set pad_token_id and eos_token_id

### Subtask:
Fix the `RuntimeError: probability tensor contains either `inf`, `nan` or element < 0` during text generation by temporarily disabling sampling and explicitly setting `pad_token_id` and `eos_token_id` in the `model.generate` call.


**Reasoning**:
To fix the `RuntimeError` during text generation, I will modify the `model.generate` call by setting `do_sample=False` and explicitly passing `tokenizer.pad_token_id` and `tokenizer.eos_token_id`.



In [ ]:
user_question = "Who is the author of Charaka Samhita?"

prompt = f"<user> {user_question} <assistant>"


inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.3,
    do_sample=False, # Disable sampling
    pad_token_id=tokenizer.pad_token_id, # Explicitly set pad_token_id
    eos_token_id=tokenizer.eos_token_id # Explicitly set eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))